# LOREVERSE Gemini 비교 점검

아래 셀 왼쪽 ▶을 누르고 InfinityFree에 설정한 모델 ID와 API 키를 입력하세요.
키는 숨김 입력으로 받아 Google Gemini API에 HTTPS 헤더로 전송합니다.
모델 조회가 성공하면 짧은 생성 요청을 한 번 보냅니다. 생성 응답 대기 시간을 55초로 설정합니다.
셀 아래에 모델 조회, 생성 결과, 시간 초과가 발생한 단계가 표시됩니다.


In [ ]:
import getpass
import http.client
import json
import re
import time
import warnings


def request_gemini(method, path, key, payload, read_timeout):
    headers = {"x-goog-api-key": key}
    body = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        body = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")

    result = {"http": 0}
    phase = "connection"
    started = time.monotonic()
    connection = http.client.HTTPSConnection("generativelanguage.googleapis.com", timeout=8)
    try:
        connection.connect()
        phase = "request_send"
        connection.request(method, path, body=body, headers=headers)
        phase = "response_headers"
        connection.sock.settimeout(read_timeout)
        response = connection.getresponse()
        result["http"] = response.status
        phase = "response_body"
        raw = response.read(1_000_001)
    except TimeoutError:
        result.update(error="GEMINI_TIMEOUT", phase=phase)
    except (OSError, http.client.HTTPException):
        result.update(error="GEMINI_NETWORK_ERROR", phase=phase)
    finally:
        connection.close()

    result["elapsed_seconds"] = round(time.monotonic() - started, 1)
    if "error" in result:
        return result, {}
    try:
        decoded = json.loads(raw)
    except ValueError:
        decoded = {}
    decoded = decoded if isinstance(decoded, dict) else {}
    status = result["http"]
    if status != 200:
        detail = decoded.get("error")
        allowed = {
            "UNAVAILABLE", "RESOURCE_EXHAUSTED", "NOT_FOUND", "PERMISSION_DENIED",
            "INVALID_ARGUMENT", "UNAUTHENTICATED", "FAILED_PRECONDITION", "INTERNAL",
        }
        provider_status = detail.get("status") if isinstance(detail, dict) else None
        if isinstance(provider_status, str) and provider_status in allowed and detail.get("code") == status:
            result["provider_status"] = provider_status
        result["error"] = f"GEMINI_HTTP_{status}"
    return result, decoded


def check_gemini():
    model = input("GEMINI_MODEL (InfinityFree의 값): ").strip()
    if not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9._-]{0,99}", model):
        return {"error": "MODEL_ID_INVALID"}

    with warnings.catch_warnings():
        warnings.simplefilter("error", getpass.GetPassWarning)
        try:
            key = getpass.getpass("GEMINI_API_KEY (숨김 입력): ")
        except (getpass.GetPassWarning, EOFError, KeyboardInterrupt):
            return {"error": "HIDDEN_INPUT_UNAVAILABLE"}
    if not key or not key.isascii() or re.search(r"[\x00-\x20\x7f]", key):
        return {"error": "KEY_INPUT_INVALID"}

    path = f"/v1beta/models/{model}"
    lookup, metadata = request_gemini("GET", path, key, None, 10)
    result = {"model_lookup": lookup}
    if "error" in lookup:
        return result
    methods = metadata.get("supportedGenerationMethods")
    lookup["supports_generation"] = isinstance(methods, list) and "generateContent" in methods
    if not lookup["supports_generation"]:
        return result

    prompt = "떠다니는 섬을 오가는 우편 배달부 세계관의 제목과 한 문장 소개를 한국어로 작성해 주세요."
    payload = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024},
    }
    generation, decoded = request_gemini("POST", f"{path}:generateContent", key, payload, 55)
    del key
    generation["valid_result"] = False
    result["generation"] = generation
    if "error" in generation:
        return result

    candidates = decoded.get("candidates")
    candidate = candidates[0] if isinstance(candidates, list) and candidates else {}
    candidate = candidate if isinstance(candidate, dict) else {}
    if candidate.get("finishReason") == "MAX_TOKENS":
        generation["error"] = "GEMINI_RESPONSE_TRUNCATED"
        return result
    content = candidate.get("content")
    parts = content.get("parts") if isinstance(content, dict) else []
    parts = parts if isinstance(parts, list) else []
    valid = candidate.get("finishReason") == "STOP" and any(
        isinstance(part, dict)
        and not part.get("thought")
        and isinstance(part.get("text"), str)
        and bool(part["text"].strip())
        for part in parts
    )
    generation["valid_result"] = valid
    if not valid:
        generation["error"] = "CHECK_INVALID_RESULT"
    return result


print(json.dumps(check_gemini(), ensure_ascii=False, indent=2))
